# Lenguajes y Autómatas II (SCD-1016)
## Sesión 05 · Recorridos del AST y Notación Polaca Inversa

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿En qué orden hay que visitar un árbol para poder calcularlo?
>
> **Objetivo de la sesión:** Recorrer un AST en postorden y producir la notación polaca inversa (RPN), el formato que usa una máquina de pila para evaluar.

**Entregable:** tu función postorden en artesano_sesion05_c.c, el harness en verde y la bitácora

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)

> ¿Dudas de C? Consulta la [guía de C](https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2/blob/main/docs/guia-c.md) antes de empezar.


## 1. ¿Qué vas a construir?

Dado un árbol, escribes su recorrido postorden separado por espacios en el buffer que te dan; el harness compara el texto exacto.

Al final de la sesión tendrás: **Una función que convierte cualquier AST en su RPN y la escribe en un buffer, sin imprimir directamente.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 05](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion05.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion05.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion05.h"

void postorden(ASTNode *n, char *buf, int cap);
```

- No imprimas con printf dentro de la función: escribe en el buffer.
- Respeta la capacidad: nunca escribas más de cap-1 caracteres más el terminador.
- Un árbol nulo debe producir cadena vacía.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion05_c.c` junto con `artesano_sesion05_c.c`
> (o `ia_sesion05_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `hoja 42` | `42` |
| 2 | `3+(5*2)` | `3 5 2 * +` |
| 3 | `arbol NULL` | `(cadena vacía)` |
| 4 | `(1+2)*(3+4)` | `1 2 + 3 4 + *` |
| 5 | `8-3` | `8 3 -` |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 05 - Recorrido postorden y RPN ===
Caso                               esperado:              obtenido:              resultado
[ 1] hoja 42                            esperado: 42           obtenido: 42           OK
[ 2] 3+(5*2)                            esperado: 3 5 2 * +    obtenido: 3 5 2 * +    OK
[ 3] arbol NULL                         esperado:              obtenido:              OK
[ 4] (1+2)*(3+4)
     esperado: 1 2 + 3 4 + *
     obtenido: 1 2 + 3 4 + *
     -> OK
[ 5] 8-3                                esperado: 8 3 -        obtenido: 8 3 -        OK

RESULTADO: 5/5 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion05.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] El recorrido respeta el orden izquierdo, derecho, raíz.
- [ ] Los tokens quedan separados por un solo espacio y sin espacio final.
- [ ] Un árbol nulo produce cadena vacía y el buffer siempre termina en '\0'.


## Paso 0 · Entorno


In [1]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


✅ Entorno listo:
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion05_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [4]:
%%writefile api_sesion05.h
#ifndef API_SESION05_H
#define API_SESION05_H

typedef enum { NODO_NUM, NODO_BINOP } TipoNodo;

typedef struct ASTNode {
    TipoNodo tipo;
    int valor;
    char op;
    struct ASTNode *izq;
    struct ASTNode *der;
} ASTNode;

void postorden(ASTNode *n, char *buf, int cap);

#endif /* API_SESION05_H */

Overwriting api_sesion05.h


In [6]:
%%writefile artesano_sesion05_c.c
#include "api_sesion05.h"
#include <stdio.h>

static void agregar_token(char *buf, int cap, const char *token) {
    int usados = 0;

    while (buf[usados] != '\0' && usados < cap - 1) {
        usados++;
    }

    if (usados >= cap - 1) {
        return;
    }

    int i = 0;

    if (usados > 0) {
        if (usados + 1 >= cap) {
            return;
        }
        buf[usados++] = ' ';
    }

    while (token[i] != '\0' && usados < cap - 1) {
        buf[usados++] = token[i++];
    }

    buf[usados] = '\0';
}

static void recorrer(ASTNode *n, char *buf, int cap) {
    if (n == NULL || cap <= 0) {
        return;
    }

    if (n->tipo == NODO_NUM) {
        char token[32];
        snprintf(token, sizeof(token), "%d", n->valor);
        agregar_token(buf, cap, token);
        return;
    }

    recorrer(n->izq, buf, cap);
    recorrer(n->der, buf, cap);

    if (n->tipo == NODO_BINOP) {
        char token[2] = {n->op, '\0'};
        agregar_token(buf, cap, token);
    }
}

void postorden(ASTNode *n, char *buf, int cap) {
    if (buf == NULL || cap <= 0) {
        return;
    }

    buf[0] = '\0';
    recorrer(n, buf, cap);
}

Overwriting artesano_sesion05_c.c


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion05_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [9]:
%%writefile ia_sesion05_c.c
#include "api_sesion05.h"
#include <stdio.h>

static void agregar_token(char *buf, int cap, const char *token) {
    int usados = 0;

    while (usados < cap - 1 && buf[usados] != '\0') {
        usados++;
    }

    if (usados >= cap - 1) {
        return;
    }

    int i = 0;

    if (usados > 0) {
        buf[usados++] = ' ';
    }

    while (token[i] != '\0' && usados < cap - 1) {
        buf[usados++] = token[i++];
    }

    buf[usados] = '\0';
}

static void recorrer(ASTNode *n, char *buf, int cap) {
    if (n == NULL || cap <= 0) {
        return;
    }

    if (n->tipo == NODO_NUM) {
        char token[32];
        snprintf(token, sizeof(token), "%d", n->valor);
        agregar_token(buf, cap, token);
        return;
    }

    recorrer(n->izq, buf, cap);
    recorrer(n->der, buf, cap);

    if (n->tipo == NODO_BINOP) {
        char token[2] = {n->op, '\0'};
        agregar_token(buf, cap, token);
    }
}

void postorden(ASTNode *n, char *buf, int cap) {
    if (buf == NULL || cap <= 0) {
        return;
    }

    buf[0] = '\0';
    recorrer(n, buf, cap);
}

Overwriting ia_sesion05_c.c


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [10]:
%%writefile harness_sesion05_c.c
/* harness_sesion05_c.c - auditoria automatica de la Sesion 05
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion05_c.c artesano_sesion05_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion05.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

static void verificar_str_largo(const char *caso, const char *esperado,
                                const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %s\n", probadas, caso);
    printf("     esperado: %s\n", esperado);
    printf("     obtenido: %s\n", obtenido ? obtenido : "(NULL)");
    printf("     -> %s\n", ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 05 - Recorrido postorden y RPN ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { ASTNode a = {NODO_NUM, 42, 0, NULL, NULL}; char b[64]; postorden(&a, b, 64); verificar_str("hoja 42", "42", b); }
    { ASTNode a = {NODO_NUM, 3, 0, NULL, NULL}; ASTNode b = {NODO_NUM, 5, 0, NULL, NULL}; ASTNode c = {NODO_NUM, 2, 0, NULL, NULL}; ASTNode m = {NODO_BINOP, 0, '*', &b, &c}; ASTNode s = {NODO_BINOP, 0, '+', &a, &m}; char buf[64]; postorden(&s, buf, 64); verificar_str("3+(5*2)", "3 5 2 * +", buf); }
    { char b[16]; postorden(NULL, b, 16); verificar_str("arbol NULL", "", b); }
    { ASTNode a = {NODO_NUM, 1, 0, NULL, NULL}; ASTNode b = {NODO_NUM, 2, 0, NULL, NULL}; ASTNode c = {NODO_NUM, 3, 0, NULL, NULL}; ASTNode d = {NODO_NUM, 4, 0, NULL, NULL}; ASTNode s1 = {NODO_BINOP, 0, '+', &a, &b}; ASTNode s2 = {NODO_BINOP, 0, '+', &c, &d}; ASTNode m = {NODO_BINOP, 0, '*', &s1, &s2}; char buf[64]; postorden(&m, buf, 64); verificar_str_largo("(1+2)*(3+4)", "1 2 + 3 4 + *", buf); }
    { ASTNode a = {NODO_NUM, 8, 0, NULL, NULL}; ASTNode b = {NODO_NUM, 3, 0, NULL, NULL}; ASTNode r = {NODO_BINOP, 0, '-', &a, &b}; char buf[32]; postorden(&r, buf, 32); verificar_str("8-3", "8 3 -", buf); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion05.md" "\n");
    return 0;
}


Writing harness_sesion05_c.c


In [11]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion05_c.c artesano_sesion05_c.c -o test_artesano && ./test_artesano

=== AUDITORIA SESION 05 - Recorrido postorden y RPN ===
Caso                               esperado:              obtenido:              resultado
[ 1] hoja 42                            esperado: 42           obtenido: 42           OK
[ 2] 3+(5*2)                            esperado: 3 5 2 * +    obtenido: 3 5 2 * +    OK
[ 3] arbol NULL                         esperado:              obtenido:              OK
[ 4] (1+2)*(3+4)
     esperado: 1 2 + 3 4 + *
     obtenido: 1 2 + 3 4 + *
     -> OK
[ 5] 8-3                                esperado: 8 3 -        obtenido: 8 3 -        OK

RESULTADO: 5/5 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion05.md


In [12]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion05_c.c ia_sesion05_c.c -o test_ia && ./test_ia

=== AUDITORIA SESION 05 - Recorrido postorden y RPN ===
Caso                               esperado:              obtenido:              resultado
[ 1] hoja 42                            esperado: 42           obtenido: 42           OK
[ 2] 3+(5*2)                            esperado: 3 5 2 * +    obtenido: 3 5 2 * +    OK
[ 3] arbol NULL                         esperado:              obtenido:              OK
[ 4] (1+2)*(3+4)
     esperado: 1 2 + 3 4 + *
     obtenido: 1 2 + 3 4 + *
     -> OK
[ 5] 8-3                                esperado: 8 3 -        obtenido: 8 3 -        OK

RESULTADO: 5/5 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion05.md


## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 5/5 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion05.md` a `auditorias/auditoria_sesion05.md` en tu fork.
2. Adjunta la salida del harness (las 5 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
